# 2025 Race Database-Surface EDA

This notebook audits the imported 2025 race-session database
surfaces in Race Telemetry Workbench and presents them as a product
story instead of a long QA appendix.

The guiding rule stays explicit:

- raw telemetry answers what was observed
- aligned replay answers when it happened
- distance-domain gained/lost analysis is a separate projection

That distinction matters because replay-quality warnings and raw
ingest coverage are related, but they are not the same thing.


## Scope And Runtime

The notebook is restricted to `year = 2025` and `session_type = 'R'`
and should only make season-wide claims when the expected 24 race
sessions are present.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "notebooks").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from notebooks import database_surface_quality_support as eda

print(f"scope: {eda.SurfaceScope().label}")
print(f"artifact dir: {eda.ARTIFACT_DIR}")


scope: 2025 race sessions
artifact dir: /Users/fabio/Workspace/race-telemetry-workbench/artifacts/2025-race-database-surface-eda


## Run The Surface Audit

The support module builds one row per 2025 race session plus the
aligned replay, context, weather, marker, and readiness tables used
below.


In [2]:
result = eda.run_analysis(write_outputs=True)

classified = result["classified"]
flag_summary = result["flag_summary"]
year_summary = result["year_summary"]
diagnostics = result["diagnostics"]
aligned_flags = result["aligned_flags"]
race_control_categories = result["race_control_categories"]
aligned_races = result["aligned_races"]
aligned_drivers = result["aligned_drivers"]
aligned_laps = result["aligned_laps"]
degraded_segments = result["degraded_segments"]
aligned_windows = result["aligned_windows"]
aligned_context_overlap = result["aligned_context_overlap"]
aligned_lap_context = result["aligned_lap_context"]
desktop_watchlist = result["desktop_watchlist"]
session_duration_coverage = result["session_duration_coverage"]
coverage_windows = result["coverage_windows"]
coverage_summary = result["coverage_summary"]
race_control_messages = result["race_control_messages"]
race_control_taxonomy_summary = result["race_control_taxonomy_summary"]
race_control_duplicates = result["race_control_duplicates"]
race_control_examples = result["race_control_examples"]
status_intervals = result["status_intervals"]
status_race_control_overlap = result["status_race_control_overlap"]
weather_summary = result["weather_summary"]
weather_transitions = result["weather_transitions"]
context_timeline_bins = result["context_timeline_bins"]
context_replay_correlation = result["context_replay_correlation"]
product_readiness = result["product_readiness"]
recommendation_summary = result["recommendation_summary"]
marker_quality = result["marker_quality"]
marker_summary = result["marker_summary"]
marker_position_examples = result["marker_position_examples"]
race_control_clustered = result["race_control_clustered"]
race_control_cluster_summary = result["race_control_cluster_summary"]

print(f"scope: {result['scope'].label}")
print(f"race sessions inspected: {len(classified)}")
print(f"sessions with at least one surface issue: {classified['has_surface_issue'].sum()}")
print(f"summary: {result['summary_path']}")


scope: 2025 race sessions
race sessions inspected: 24
sessions with at least one surface issue: 24
summary: /Users/fabio/Workspace/race-telemetry-workbench/docs/data-quality/2025-race-database-surface-eda-summary.md


In [3]:
import pandas as pd
from IPython.display import HTML, Markdown, display

def show_table(df, columns=None, sort_by=None, ascending=False, limit=10, title=None):
    table = df.copy()
    if sort_by and sort_by in table.columns:
        table = table.sort_values(sort_by, ascending=ascending)
    if columns:
        keep = [column for column in columns if column in table.columns]
        if keep:
            table = table[keep]
    if title:
        display(Markdown(f"### {title}"))
    display(table.head(limit))

def metric_row(cards):
    html_cards = []
    for label, value, detail in cards:
        html_cards.append(
            f'''
            <div style="flex:1; min-width:180px; border:1px solid #d6dde6; border-radius:14px; padding:14px 16px; background:linear-gradient(180deg,#fbfdff 0%,#f2f6fa 100%);">
              <div style="font-size:12px; text-transform:uppercase; letter-spacing:0.08em; color:#5b677a; margin-bottom:6px;">{label}</div>
              <div style="font-size:28px; font-weight:700; color:#1f2933; line-height:1.1;">{value}</div>
              <div style="font-size:13px; color:#52606d; margin-top:6px;">{detail}</div>
            </div>
            '''
        )
    display(
        HTML(
            '<div style="display:flex; gap:12px; flex-wrap:wrap; margin:10px 0 18px 0;">'
            + "".join(html_cards)
            + "</div>"
        )
    )

def callout(title: str, body: str, accent: str = "#1f6feb"):
    display(
        HTML(
            f'''
            <div style="border-left:5px solid {accent}; background:#f7fafc; padding:12px 16px; margin:10px 0 16px 0; border-radius:10px;">
              <div style="font-weight:700; color:#102a43; margin-bottom:4px;">{title}</div>
              <div style="color:#334e68; line-height:1.45;">{body}</div>
            </div>
            '''
        )
    )


## Quick Read

This notebook starts with a product question rather than a database
question: if a human opened the imported 2025 season today, which
surfaces look broadly trustworthy and which ones need visible caveats?


In [4]:
session_count = len(classified)
sessions_with_issues = int(classified["has_surface_issue"].sum())
issue_pct = sessions_with_issues / session_count * 100 if session_count else 0
worst_surface = flag_summary.sort_values("sessions", ascending=False).iloc[0]
top_recommendation = recommendation_summary.sort_values("sessions", ascending=False).iloc[0]

metric_row([
    ("Race sessions", f"{session_count:,}", result["scope"].label),
    ("Sessions with issues", f"{sessions_with_issues:,}", f"{issue_pct:.1f}% carry at least one surface warning"),
    ("Most common flag", worst_surface["surface_flag"], f"{int(worst_surface['sessions'])} sessions"),
    ("Main recommendation", top_recommendation["final_recommendation"], f"{int(top_recommendation['sessions'])} sessions"),
])

callout(
    "Read this notebook by domain",
    "Raw ingest coverage, aligned replay quality, and product readiness are separated on purpose. Treating them as the same thing usually leads to misleading conclusions.",
)


### Coverage Across The Imported Surfaces

This is the fastest season-wide view of what exists before we judge quality or product risk.

![Coverage Across The Imported Surfaces](../artifacts/2025-race-database-surface-eda/figures/surface_availability_heatmap.svg)


### Which Surfaces Drive The Warnings

A session can be replay-safe and still show a surface issue worth documenting or labeling.

![Which Surfaces Drive The Warnings](../artifacts/2025-race-database-surface-eda/figures/surface_issue_counts.svg)


In [5]:
show_table(
    classified,
    columns=[
        "year",
        "event_name",
        "driver_count",
        "lap_rows",
        "telemetry_samples",
        "position_samples",
        "aligned_samples",
        "surface_issue_count",
    ],
    sort_by="surface_issue_count",
    ascending=False,
    limit=8,
    title="Sessions With The Most Surface Warnings",
)


### Sessions With The Most Surface Warnings

,year,event_name,driver_count,lap_rows,telemetry_samples,position_samples,aligned_samples,surface_issue_count
0,2025,Australian Grand Prix,20,927,372477,380930,999583,3
10,2025,Austrian Grand Prix,20,1126,307177,314417,821801,3
21,2025,Las Vegas Grand Prix,20,886,328580,334193,874472,3
20,2025,São Paulo Grand Prix,20,1251,369890,372745,983789,3
16,2025,Azerbaijan Grand Prix,20,968,401811,403191,1074586,3
14,2025,Dutch Grand Prix,20,1364,411922,426422,1119107,3
11,2025,British Grand Prix,20,825,350061,359209,943107,3
12,2025,Belgian Grand Prix,20,879,385605,394553,1035545,3


## Coverage First, Then Quality

A human EDA usually asks whether the low-level surfaces exist before
it asks whether the higher-level replay model is trustworthy. This
section keeps that order explicit.


### Raw Ingestion Cadence

The point here is not a perfect nominal frequency, but whether certain streams or races drift enough to explain later replay degradation.

![Raw Ingestion Cadence](../artifacts/2025-race-database-surface-eda/figures/ingestion_frequency_by_stream.svg)


### Active Replay Coverage By Surface

This chart narrows the focus from whole-session presence to the windows that matter most for playback.

![Active Replay Coverage By Surface](../artifacts/2025-race-database-surface-eda/figures/surface_active_coverage_heatmap.svg)


### Coverage Windows Across The Session

The temporal spread matters because some sessions have coverage that is technically present but poorly placed for live replay use.

![Coverage Windows Across The Session](../artifacts/2025-race-database-surface-eda/figures/surface_coverage_windows.svg)


In [6]:
show_table(
    diagnostics,
    columns=[
        "event_name",
        "driver_code",
        "stream_name",
        "estimated_frequency_hz",
        "sample_count",
    ],
    sort_by="estimated_frequency_hz",
    ascending=True,
    limit=12,
    title="Selected Ingestion Diagnostics",
)
show_table(
    coverage_summary,
    limit=len(coverage_summary),
    title="Coverage Summary By Surface",
)


### Selected Ingestion Diagnostics

,event_name,driver_code,stream_name,estimated_frequency_hz,sample_count
773,Mexico City Grand Prix,LAW,raw_location_telemetry,3.846154,1815
455,British Grand Prix,LAW,raw_location_telemetry,3.846154,402
637,Azerbaijan Grand Prix,ANT,raw_location_telemetry,3.846154,21117
445,British Grand Prix,BOR,raw_location_telemetry,3.846154,2107
33,Australian Grand Prix,SAI,raw_location_telemetry,3.846154,418
671,Azerbaijan Grand Prix,VER,raw_location_telemetry,3.846154,21032
633,Azerbaijan Grand Prix,ALB,raw_location_telemetry,3.846154,21267
635,Azerbaijan Grand Prix,ALO,raw_location_telemetry,3.846154,21335
667,Azerbaijan Grand Prix,STR,raw_location_telemetry,3.846154,21406
435,Austrian Grand Prix,VER,raw_location_telemetry,3.846154,335


### Coverage Summary By Surface

,surface,sessions,median_coverage_ratio,median_active_coverage_ratio,min_active_coverage_ratio,sessions_starting_after_active,sessions_ending_before_active_end
0,raw telemetry,24,0.591998,1.000000,1.000000,0,0
1,raw position,24,0.592002,0.415679,0.274514,0,24
2,aligned replay,24,0.591973,1.000000,1.000000,0,0
3,weather,24,0.990176,1.000000,0.995103,0,0
4,track status,24,0.740655,0.615276,0.000000,0,20
5,session status,24,0.999171,1.000000,1.000000,0,0
6,race control,24,0.630772,0.454820,0.317823,0,24


## Replay Quality Is Its Own Story

`aligned_telemetry_10hz` is a replay-oriented derived surface. It is
useful, but it should never be mistaken for raw truth. The next few
charts focus on how replay degrades, where it degrades, and whether
those degradations line up with obvious race context.


In [7]:
callout(
    "Time-domain rule",
    "Replay-quality warnings are time-domain diagnostics. They answer when the replay model was forced to interpolate, age out, or degrade, not where a driver gained or lost time.",
    accent="#b54708",
)


### Replay Quality By Driver And Race

This is the season map for non-OK aligned rows.

![Replay Quality By Driver And Race](../artifacts/2025-race-database-surface-eda/figures/aligned_driver_non_ok_heatmap.svg)


### What Degraded Replay Windows Look Like

The strip view is closer to how the desktop player experiences replay problems than an aggregate percentage table.

![What Degraded Replay Windows Look Like](../artifacts/2025-race-database-surface-eda/figures/aligned_quality_replay_strips.svg)


### Do Degraded Windows Coincide With Context-Heavy Moments?

This correlation is descriptive, not causal, but it is useful for deciding whether warnings should mention incidents, pit phases, or neutralization periods.

![Do Degraded Windows Coincide With Context-Heavy Moments?](../artifacts/2025-race-database-surface-eda/figures/aligned_context_overlap.svg)


In [8]:
show_table(
    aligned_races,
    columns=["event_name", "aligned_rows", "non_ok_rows", "non_ok_pct"],
    sort_by="non_ok_pct",
    ascending=False,
    limit=10,
    title="Races With The Highest Non-OK Replay Share",
)
show_table(
    desktop_watchlist,
    columns=[
        "event_name",
        "driver_code",
        "window_start_ms",
        "window_end_ms",
        "non_ok_pct",
        "dominant_family",
    ],
    sort_by="non_ok_pct",
    ascending=False,
    limit=12,
    title="Replay Windows Worth Watching In Product QA",
)


### Races With The Highest Non-OK Replay Share

,event_name,aligned_rows,non_ok_rows,non_ok_pct
0,Dutch Grand Prix,1119107,31300,2.796873
5,Austrian Grand Prix,821801,22295,2.712944
3,British Grand Prix,943107,23768,2.520181
1,Hungarian Grand Prix,1134521,27436,2.418289
2,Belgian Grand Prix,1035545,24193,2.336258
4,Monaco Grand Prix,1131575,23043,2.036365
6,Mexico City Grand Prix,1059385,20890,1.971899
19,Italian Grand Prix,820417,15478,1.886602
8,Abu Dhabi Grand Prix,1043228,19164,1.836991
12,Miami Grand Prix,955079,17542,1.836707


### Replay Windows Worth Watching In Product QA

,event_name,driver_code,non_ok_pct
464,Austrian Grand Prix,ANT,4.436860
465,Austrian Grand Prix,VER,4.436860
456,British Grand Prix,BOR,3.398927
8,Dutch Grand Prix,STR,2.816211
0,Dutch Grand Prix,ALO,2.815165
9,Dutch Grand Prix,TSU,2.814262
6,Dutch Grand Prix,OCO,2.814025
2,Dutch Grand Prix,COL,2.813835
5,Dutch Grand Prix,LAW,2.812743
7,Dutch Grand Prix,SAI,2.812505


## Context Surfaces Should Tell A Story Too

Race control, status, and weather are not just auxiliary tables.
They are the narrative layer that explains why certain replay windows
deserve labels, chips, or richer UI treatment.


### Race-Control Message Mix

This is the category balance check before building incident stories on top of the text feed.

![Race-Control Message Mix](../artifacts/2025-race-database-surface-eda/figures/race_control_category_mix.svg)


### Taxonomy Depth

The deterministic taxonomy should cover the common phrase families before clustering is allowed to explain the leftovers.

![Taxonomy Depth](../artifacts/2025-race-database-surface-eda/figures/race_control_taxonomy_mix.svg)


### Status Timeline Strips

Status intervals are inherently temporal, so a strip chart communicates them better than a flat table.

![Status Timeline Strips](../artifacts/2025-race-database-surface-eda/figures/status_timeline_strips.svg)


### Where The Session Context Peaks

This reveals whether context density is concentrated in a few violent stretches or spread across the race.

![Where The Session Context Peaks](../artifacts/2025-race-database-surface-eda/figures/context_timeline_density.svg)


In [9]:
show_table(
    race_control_taxonomy_summary,
    columns=["taxonomy", "messages", "sessions"],
    sort_by="messages",
    ascending=False,
    limit=10,
    title="Top Race-Control Taxonomy Buckets",
)
show_table(
    context_replay_correlation,
    limit=len(context_replay_correlation),
    title="Context / Replay Correlation Summary",
)
show_table(
    race_control_duplicates,
    columns=["event_name", "taxonomy", "message", "duplicates"],
    sort_by="duplicates",
    ascending=False,
    limit=10,
    title="Repeated Race-Control Messages",
)


### Top Race-Control Taxonomy Buckets

,taxonomy,messages,sessions
1,flags,1008,24
3,other,431,24
2,investigations_noted,318,23
0,drs,139,24
7,safety_car,92,18
5,pit_entry_exit,89,24
4,penalties,76,19
6,red_flag,25,24


### Context / Replay Correlation Summary

,bin_bucket,bins,median_degraded_window_rate,p95_degraded_window_rate,median_max_non_ok_pct,p95_max_non_ok_pct,severe_driver_windows
0,all_bins,466,50.000000,78.517615,4.666667,9.000000,133
1,context_event_bins,225,50.000000,80.000000,5.000000,9.000000,97
2,no_context_event_bins,241,40.880503,70.000000,4.666667,9.000000,36
3,race_control_incident_bins,168,50.000000,80.000000,5.000000,8.666667,58
4,status_bins,96,50.000000,82.500000,5.000000,9.833333,78
5,rainfall_transition_bins,15,40.000000,72.000000,6.666667,8.966667,0


### Repeated Race-Control Messages

,event_name,taxonomy
297,Belgian Grand Prix,flags
327,British Grand Prix,flags
366,British Grand Prix,flags
298,Belgian Grand Prix,flags
88,Australian Grand Prix,flags
823,Monaco Grand Prix,flags
459,Dutch Grand Prix,flags
819,Monaco Grand Prix,flags
191,Azerbaijan Grand Prix,flags
192,Azerbaijan Grand Prix,flags


## Weather And Marker Geometry

These surfaces are visually rich and easy to misunderstand in plain
tables, so they benefit the most from being chart-led.


### Weather Cadence And Change Intensity

This separates sparse weather feeds from races that actually experienced meaningful transitions.

![Weather Cadence And Change Intensity](../artifacts/2025-race-database-surface-eda/figures/weather_cadence_jumps.svg)


### Weather Panels For High-Change Sessions

The panels make it obvious whether the imported weather stream would support a believable timeline overlay.

![Weather Panels For High-Change Sessions](../artifacts/2025-race-database-surface-eda/figures/weather_trend_panels.svg)


### Circuit Marker Quality

Marker counts alone are not enough; we also need to know whether coordinates are plausible against imported traces.

![Circuit Marker Quality](../artifacts/2025-race-database-surface-eda/figures/circuit_marker_quality_summary.svg)


### Marker Overlay Examples

This is the geometry sanity check before the product treats corners or marshal lights as trustworthy annotation anchors.

![Marker Overlay Examples](../artifacts/2025-race-database-surface-eda/figures/circuit_marker_overlay_examples.svg)


In [10]:
show_table(
    weather_summary,
    columns=[
        "event_name",
        "weather_samples",
        "max_gap_ms",
        "large_gap_flag",
        "rainfall_transitions",
    ],
    sort_by="max_gap_ms",
    ascending=False,
    limit=10,
    title="Weather Sessions To Inspect",
)
show_table(
    marker_quality[marker_quality["marker_coordinate_issue"] != "none"],
    columns=[
        "event_name",
        "marker_type",
        "marker_number",
        "marker_letter",
        "marker_coordinate_issue",
    ],
    sort_by="event_name",
    ascending=True,
    limit=12,
    title="Marker Coordinate Exceptions",
)


### Weather Sessions To Inspect

,event_name,max_gap_ms,large_gap_flag,rainfall_transitions
0,Abu Dhabi Grand Prix,60224.0,False,0
6,British Grand Prix,60220.0,False,4
5,Belgian Grand Prix,60200.0,False,12
14,Las Vegas Grand Prix,60140.0,False,0
4,Bahrain Grand Prix,60137.0,False,0
19,São Paulo Grand Prix,60127.0,False,0
1,Australian Grand Prix,60092.0,False,17
22,Spanish Grand Prix,60082.0,False,0
13,Japanese Grand Prix,60082.0,False,0
2,Austrian Grand Prix,60081.0,False,0


### Marker Coordinate Exceptions

,event_name,marker_type,marker_number,marker_letter,marker_coordinate_issue


## Product Readiness

After coverage, replay, context, and geometry, the notebook can
finally translate QA evidence into product-facing recommendations.


### Readiness Across Product Lenses

The dashboard keeps catalog, raw-stream, replay, context, and circuit-context readiness separate.

![Readiness Across Product Lenses](../artifacts/2025-race-database-surface-eda/figures/product_readiness_dashboard.svg)


### Recommendation Mix

This is the action view: no action, UI label, inspect, reimport, or schema/importer work.

![Recommendation Mix](../artifacts/2025-race-database-surface-eda/figures/product_recommendation_summary.svg)


In [11]:
show_table(
    product_readiness,
    columns=[
        "event_name",
        "catalog_readiness",
        "raw_stream_readiness",
        "replay_readiness",
        "context_readiness",
        "circuit_context_readiness",
        "final_recommendation",
    ],
    sort_by="final_recommendation",
    ascending=True,
    limit=12,
    title="Session-Level Product Readiness",
)
show_table(
    recommendation_summary,
    limit=len(recommendation_summary),
    title="Recommendation Totals",
)


### Session-Level Product Readiness

,event_name,catalog_readiness,raw_stream_readiness,replay_readiness,context_readiness,circuit_context_readiness,final_recommendation
0,Australian Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,label_in_ui
21,Las Vegas Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,label_in_ui
20,São Paulo Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,label_in_ui
19,Mexico City Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,ready,label_in_ui
18,United States Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,ready,label_in_ui
17,Singapore Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,label_in_ui
16,Azerbaijan Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,label_in_ui
15,Italian Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,label_in_ui
14,Dutch Grand Prix,ready_with_warnings,ready_with_warnings,partial,ready_with_warnings,ready,label_in_ui
13,Hungarian Grand Prix,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready_with_warnings,ready,label_in_ui


### Recommendation Totals

,final_recommendation,sessions,max_readiness_score,affected_drivers,affected_rows,severe_replay_windows,marker_coordinate_issues,schema_importer_follow_up_sessions
0,no_action,0,0,0,0,0,0,0
1,label_in_ui,24,2,476,458158,133,0,24
2,inspect,0,0,0,0,0,0,0
3,reimport,0,0,0,0,0,0,0
4,schema_importer_change,0,0,0,0,0,0,0


## Appendix: Text Clusters And Supporting Tables

These remain useful, but they belong after the main visual argument.


### Race-Control Text Clusters

Clustering is a supplement to the deterministic taxonomy, not a replacement for it.

![Race-Control Text Clusters](../artifacts/2025-race-database-surface-eda/figures/race_control_text_clusters.svg)


In [12]:
show_table(
    race_control_cluster_summary,
    columns=["text_cluster", "cluster_terms", "messages"],
    sort_by="messages",
    ascending=False,
    limit=12,
    title="Largest Text Clusters",
)
show_table(
    year_summary,
    limit=len(year_summary),
    title="Year-Level Coverage Check",
)


### Largest Text Clusters

,text_cluster,cluster_terms,messages
3,3,"blue flag, waved, timed, waved blue, blue, fla...",480
0,0,"car, safety car, safety, involving car, penalt...",405
4,4,"lap, turn lap, deleted track, deleted, limits ...",324
1,1,"clear, clear track, sector, track sector, trac...",255
5,5,"yellow track, yellow, sector, track sector, do...",245
7,7,"turn incident, involving cars, cars, incident,...",217
2,2,"drs enabled, enabled, drs, enabled zone, zone,...",74
6,6,"pit exit, exit, pit, green light, green, light...",72
8,8,"drs disabled, disabled, drs, disabled zone, zo...",63
9,9,"slippery track, slippery, surface slippery, su...",43


### Year-Level Coverage Check

,year,sessions,sessions_with_issue,median_surface_issue_count,telemetry_samples,position_samples,aligned_samples,weather_samples,race_control_messages,issue_pct
0,2025,24,24,2.0,9113457,9314091,24477559,3781,2178,100.0


## Outputs

Primary outputs:

- `docs/data-quality/2025-race-database-surface-eda-summary.md`
- `artifacts/2025-race-database-surface-eda/skrub_2025_race_database_surface_report.html`
- `artifacts/2025-race-database-surface-eda/tables/*.csv`
- `artifacts/2025-race-database-surface-eda/tables/*.parquet`
- `artifacts/2025-race-database-surface-eda/figures/*.svg`

The next useful slice remains explicit: add the distance-domain
projection and keep any “where time was gained or lost” analysis out
of this notebook until that surface exists.
